In [1]:
from GARCH import get_garch_output
from GARCH import garch_1_1_volatility_forecast
from GARCH import deterministic_portfolio_return_forecasts

In [2]:
#past returns training data
#default values
interval = "1d"
period = "5y"
price = "Close"

#optional GARCH starting para
#default for omega 0.05 * var(returns)
starting_para = {
    "alpha": 0.5,
    "beta": 0.9,
    "omega": 0.05
}

assets = ["NVO", "AAPL", "SAP", "SI", "NVDA"]

output = get_garch_output(tickers=assets, interval=interval, period=period, price=price, starting_para=starting_para)

In [3]:
output["NVO"]

{'alpha': np.float64(0.13711487458238858),
 'beta': np.float64(0.7983332463505826),
 'omega': np.float64(0.5080521607043862),
 'implied unconditional variance': np.float64(7.8704472750799255),
 'sample unconditional variance': np.float64(5.884447497207841),
 'expected return': np.float64(0.017201853327737555),
 'return last period': np.float64(-1.023962581592013),
 'conditional vola t+1': np.float64(5.162974581556379),
 'past_z_values': Date
 2021-04-20 00:00:00-04:00    0.098783
 2021-04-21 00:00:00-04:00    0.184868
 2021-04-22 00:00:00-04:00   -0.020106
 2021-04-23 00:00:00-04:00   -0.025331
 2021-04-26 00:00:00-04:00    0.007540
                                ...   
 2026-04-13 00:00:00-04:00    0.324664
 2026-04-14 00:00:00-04:00    0.942663
 2026-04-15 00:00:00-04:00    0.731351
 2026-04-16 00:00:00-04:00    0.042997
 2026-04-17 00:00:00-04:00   -0.181209
 Name: Close, Length: 1255, dtype: float64}

In [4]:
#volatility forecast
# default forecast period
forecast_period = 5
garch_1_1_volatility_forecast(output, forecast_period)

In [6]:
#compare to arch package
from arch import arch_model
from GARCH import compute_log_returns
import yfinance as yf

nvo_returns = yf.Ticker("NVO").history(period=period, interval=interval)[price]
log_returns = compute_log_returns(nvo_returns)

model = arch_model(log_returns[0], p=1, q=1).fit()
print(model.summary())


Iteration:      1,   Func. Count:      6,   Neg. LLF: 5851.512410475805
Iteration:      2,   Func. Count:     14,   Neg. LLF: 40590.06693486521
Iteration:      3,   Func. Count:     21,   Neg. LLF: 3058.707091627742
Iteration:      4,   Func. Count:     28,   Neg. LLF: 2843.2618737024645
Iteration:      5,   Func. Count:     34,   Neg. LLF: 2840.0268767870143
Iteration:      6,   Func. Count:     40,   Neg. LLF: 2859.5803988328767
Iteration:      7,   Func. Count:     46,   Neg. LLF: 2836.871856593274
Iteration:      8,   Func. Count:     51,   Neg. LLF: 2836.864572235384
Iteration:      9,   Func. Count:     56,   Neg. LLF: 2836.864437373188
Iteration:     10,   Func. Count:     61,   Neg. LLF: 2836.864436797444
Optimization terminated successfully    (Exit mode 0)
            Current function value: 2836.864436797444
            Iterations: 10
            Function evaluations: 61
            Gradient evaluations: 10
                     Constant Mean - GARCH Model Results            

In [7]:
import numpy as np
np.sqrt(model.forecast(horizon=forecast_period).variance.values[0])

array([2.25366725, 2.30125395, 2.34588925, 2.38782271, 2.4272735 ])

In [8]:
#forecast scenario based returns
#define scenarios for example to show path dependencies on total returns
s1 = [0.3, -0.2, -0.1, 0.2]
s2 = [-0.2, -0.1, 0.2, 0.3]
s3 = [0.3, 0.2, -0.1, -0.2]
scenarios = [s1, s2, s3]

#define asset (main asset) for which these scenarios are assumed
main_asset = "NVO"

#define portfolio weights for each asset
weights = {
    "NVO": 0.3,
    "AAPL": 0.2,
    "SAP": 0.1,
    "SI": 0.2,
    "NVDA": 0.1
    }

deterministic_portfolio_return_forecasts(garch_output = output, scenarios=scenarios, main_asset=main_asset, asset_weights=weights)